## Aggregating (Average) ETa data into Monthly Scale

In [ ]:
import glob
import os
import re
import pandas as pd
import xarray as xr
import rioxarray as rxr

# 1. Define the input folder containing your ETa GeoTIFFs and the output folder 
#    for monthly averages.
input_folder = r"./data/raw/ETa_Rasters"
output_folder = r"./data/processed/ETa/Monthly"
os.makedirs(output_folder, exist_ok=True)

# 2. Gather all TIFF files.
tif_files = sorted(glob.glob(os.path.join(input_folder, "*.tif")))

# 3. Function to extract date from filename.
#    Assumes filenames like: ETa_YYYY-MM-DD.tif. Modify if needed.
def extract_date_from_filename(fname):
    base = os.path.basename(fname)
    match = re.search(r"(\d{4}-\d{2}-\d{2})", base)
    if match:
        return pd.to_datetime(match.group(1))
    else:
        print(f"Warning: No valid date found in filename {fname}")
        return None

# 4. Read all rasters into a single xarray DataArray with 'time' as a dimension.
da_list = []
for f in tif_files:
    date = extract_date_from_filename(f)
    if date is None:
        print(f"Skipping file {f} because date could not be parsed.")
        continue
    # Read the raster (masked=True handles nodata)
    da = rxr.open_rasterio(f, masked=True).squeeze()  # Remove any extra dimensions
    da = da.expand_dims(time=[date])
    da_list.append(da)

if not da_list:
    raise ValueError("No valid rasters were found or parsed. Check your filenames.")

combined_da = xr.concat(da_list, dim="time")
combined_da = combined_da.sortby("time")  # Ensure chronological order

# 5. Aggregate monthly by averaging. 
#    The resolution stays at 30 m as the data is not resampled.
monthly_means = combined_da.resample(time="MS").mean()
# "MS" = Month Start. You can also use "M" for month-end if preferred.
for single_time in monthly_means.time:
    # Format the time string to "YYYY_Mmm" (e.g., "2018_Jan")
    t_str = pd.to_datetime(single_time.values).strftime("%Y_%b")
    out_name = os.path.join(output_folder, f"ETa_{t_str}.tif")
    
    # Select this month's DataArray and squeeze out the time dimension
    da_month = monthly_means.sel(time=single_time).squeeze()
    
    # Save to GeoTIFF
    da_month.rio.to_raster(out_name)
    print(f"Saved monthly average for {t_str} to {out_name}")

print("Monthly aggregation complete!")

## Resampling to 5 m and Pixel Alignment with DEM

In [ ]:
import os
import glob
from osgeo import gdal

# ------------------------------------------------------------------------------
# 1. Specify your DEM path (including filename)
# ------------------------------------------------------------------------------
dem_path = r"./data/raw/Topographic_Features/DEM.tif"

# ------------------------------------------------------------------------------
# 2. Specify your ETa imagery folder
#    (change the path to wherever your ETa .tif files are stored)
# ------------------------------------------------------------------------------
eta_dir = r"./data/processed/ETa/Monthly"

# ------------------------------------------------------------------------------
# 3. Specify the output directory for your resampled ETa files
# ------------------------------------------------------------------------------
out_dir = r"./data/processed/ETa/Resampled_5m"
os.makedirs(out_dir, exist_ok=True)

eta_files = glob.glob(os.path.join(eta_dir, "*.tif"))

dem_ds = gdal.Open(dem_path)
dem_proj = dem_ds.GetProjection()  # Get DEM’s projection
dem_ds = None  # Close DEM

for eta_file in eta_files:
    out_file = os.path.join(out_dir, os.path.basename(eta_file))

    warp_options = gdal.WarpOptions(
        format='GTiff',
        xRes=10,               # Desired resolution in X direction
        yRes=10,               # Desired resolution in Y direction
        dstSRS=dem_proj,      # Match the DEM’s projection
        resampleAlg='near',   # Resampling method = nearest neighbor
        targetAlignedPixels=True
    )

    gdal.Warp(
        destNameOrDestDS=out_file,
        srcDSOrSrcDSTab=eta_file,
        options=warp_options
    )

print("Resampling complete!")